# 03 - Qdrant Indexing

Indexa embeddings visuales de crops GT en Qdrant. `class_id` y `class_name` se guardan solo como payload.

In [1]:
%pip install numpy pandas Pillow torch torchvision open_clip_torch qdrant-client

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
import uuid
import numpy as np
import pandas as pd
from PIL import Image
import torch
import open_clip
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

OUTPUTS_DIR = PROJECT_ROOT / "outputs"
COLLECTION_NAME = "xray_clip_gt_rois"
USE_IN_MEMORY_QDRANT = True
QDRANT_URL = "http://localhost:6333"

MODEL_NAME = "ViT-B-32"
PRETRAINED = "laion2b_s34b_b79k"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CLASS_PROMPT_NAMES = {
    "0": "gun",
    "1": "knife",
    "2": "pliers",
    "3": "scissors",
    "4": "wrench",
}

PROMPT_TEMPLATES = [
    "a baggage x-ray image of a {}",
    "an x-ray crop containing a {}",
    "a prohibited item: {}",
    "a security x-ray image showing a {}",
    "a cropped x-ray baggage image of a {}",
]

c:\Users\Juan Ramirez\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
embeddings = np.load(OUTPUTS_DIR / "embeddings.npy").astype("float32")
metadata = pd.read_csv(OUTPUTS_DIR / "embeddings_metadata.csv")
all_metadata = pd.read_csv(OUTPUTS_DIR / "crops_metadata.csv")

metadata["class_id"] = metadata["class_id"].astype(str)
metadata["class_name"] = metadata["class_id"].map(CLASS_PROMPT_NAMES)
all_metadata["class_id"] = all_metadata["class_id"].astype(str)
all_metadata["class_name"] = all_metadata["class_id"].map(CLASS_PROMPT_NAMES)

assert len(metadata) == embeddings.shape[0]
assert set(metadata["split"].unique()).issubset({"train", "valid"}), "Qdrant index must contain only train+valid crops"
assert not (metadata["split"] == "test").any(), "Test crops must not be indexed"
assert not np.isnan(embeddings).any()
assert np.allclose(np.linalg.norm(embeddings, axis=1), 1.0, atol=1e-4)
print(embeddings.shape)
metadata.head()

(7947, 512)


,crop_id,split,image_path,label_path,crop_path,class_id,class_name,x1,y1,x2,y2,bbox_width,bbox_height,bbox_area,image_width,image_height,relative_area
0,train_009000_jpg.rf.8c46e1aa5b46a0ad24ee4bcb29...,train,C:\Users\Juan Ramirez\.cache\kagglehub\dataset...,C:\Users\Juan Ramirez\.cache\kagglehub\dataset...,data\crops\train\train_009000_jpg.rf.8c46e1aa5...,2,pliers,258,210,286,241,28,31,868,416,416,0.005016
1,train_009002_jpg.rf.18bf80f2cfdb51f853da15019f...,train,C:\Users\Juan Ramirez\.cache\kagglehub\dataset...,C:\Users\Juan Ramirez\.cache\kagglehub\dataset...,data\crops\train\train_009002_jpg.rf.18bf80f2c...,2,pliers,219,125,245,157,26,32,832,416,416,0.004808
2,train_009003_jpg.rf.46963402c4cb6f46a47e508b89...,train,C:\Users\Juan Ramirez\.cache\kagglehub\dataset...,C:\Users\Juan Ramirez\.cache\kagglehub\dataset...,data\crops\train\train_009003_jpg.rf.46963402c...,2,pliers,153,145,199,171,46,26,1196,416,416,0.006911
3,train_009007_jpg.rf.a5143afbb0c741f3b60fc72403...,train,C:\Users\Juan Ramirez\.cache\kagglehub\dataset...,C:\Users\Juan Ramirez\.cache\kagglehub\dataset...,data\crops\train\train_009007_jpg.rf.a5143afbb...,2,pliers,154,105,180,187,26,82,2132,416,416,0.012320
4,train_009012_jpg.rf.bc99877ade8754d2be89119361...,train,C:\Users\Juan Ramirez\.cache\kagglehub\dataset...,C:\Users\Juan Ramirez\.cache\kagglehub\dataset...,data\crops\train\train_009012_jpg.rf.bc99877ad...,2,pliers,322,144,347,178,25,34,850,416,416,0.004912


In [4]:
client = QdrantClient(":memory:") if USE_IN_MEMORY_QDRANT else QdrantClient(url=QDRANT_URL)
vector_size = embeddings.shape[1]

if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=vector_size, distance=Distance.COSINE),
)

payload_columns = [
    "crop_id", "split", "image_path", "crop_path", "class_id", "class_name",
    "x1", "y1", "x2", "y2", "bbox_area", "relative_area",
]

points = []
for idx, row in metadata.iterrows():
    payload = {col: row[col].item() if hasattr(row[col], "item") else row[col] for col in payload_columns}
    points.append(PointStruct(
        id=str(uuid.uuid5(uuid.NAMESPACE_URL, str(row["crop_id"]))),
        vector=embeddings[idx].tolist(),
        payload=payload,
    ))

client.upsert(collection_name=COLLECTION_NAME, points=points)
print(f"Indexed {len(points)} visual crop embeddings in {COLLECTION_NAME}")

Indexed 7947 visual crop embeddings in xray_clip_gt_rois


In [5]:
model, _, preprocess = open_clip.create_model_and_transforms(MODEL_NAME, pretrained=PRETRAINED, device=DEVICE)
tokenizer = open_clip.get_tokenizer(MODEL_NAME)
model.eval()

crop_id_to_index = {crop_id: idx for idx, crop_id in enumerate(metadata["crop_id"].tolist())}
all_crop_id_to_path = dict(zip(all_metadata["crop_id"], all_metadata["crop_path"]))


def normalize_np(vector):
    vector = np.asarray(vector, dtype="float32")
    return vector / np.linalg.norm(vector)


def load_preprocessed_crop(relative_crop_path):
    crop_path = Path(relative_crop_path)
    if not crop_path.is_absolute():
        crop_path = PROJECT_ROOT / crop_path
    with Image.open(crop_path) as image:
        image = image.convert("RGB")
        return preprocess(image)


def encode_image_crop(relative_crop_path):
    with torch.no_grad():
        image = load_preprocessed_crop(relative_crop_path).unsqueeze(0).to(DEVICE)
        features = model.encode_image(image)
        features = features / features.norm(dim=-1, keepdim=True)
    return features[0].cpu().numpy().astype("float32")


def qdrant_search(query_vector, top_k=5, query_filter=None):
    query_vector = normalize_np(query_vector).tolist()
    if hasattr(client, "search"):
        return client.search(
            collection_name=COLLECTION_NAME,
            query_vector=query_vector,
            query_filter=query_filter,
            limit=top_k,
            with_payload=True,
        )

    response = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        query_filter=query_filter,
        limit=top_k,
        with_payload=True,
    )
    return response.points if hasattr(response, "points") else response


def encode_text_query(query_text):
    with torch.no_grad():
        tokens = tokenizer([query_text]).to(DEVICE)
        features = model.encode_text(tokens)
        features = features / features.norm(dim=-1, keepdim=True)
    return features[0].cpu().numpy().astype("float32")


def encode_class_name(class_name):
    if class_name not in CLASS_PROMPT_NAMES.values():
        raise ValueError(f"Use natural class names only: {list(CLASS_PROMPT_NAMES.values())}")
    prompts = [template.format(class_name) for template in PROMPT_TEMPLATES]
    with torch.no_grad():
        tokens = tokenizer(prompts).to(DEVICE)
        prompt_embeddings = model.encode_text(tokens)
        prompt_embeddings = prompt_embeddings / prompt_embeddings.norm(dim=-1, keepdim=True)
        final_text_embedding = prompt_embeddings.mean(dim=0)
        final_text_embedding = final_text_embedding / final_text_embedding.norm()
    return final_text_embedding.cpu().numpy().astype("float32")


def format_hits(hits):
    rows = []
    for rank, hit in enumerate(hits, start=1):
        payload = hit.payload or {}
        rows.append({"rank": rank, "score": hit.score, **payload})
    return pd.DataFrame(rows)


def retrieve_by_text(query_text, top_k=5):
    query_vector = encode_text_query(query_text)
    return format_hits(qdrant_search(query_vector, top_k=top_k))


def retrieve_by_class_name(class_name, top_k=5):
    if str(class_name) in CLASS_PROMPT_NAMES.keys():
        raise ValueError("Do not query CLIP with class_id. Use class_name such as 'gun'.")
    query_vector = encode_class_name(class_name)
    return format_hits(qdrant_search(query_vector, top_k=top_k))


def retrieve_by_image(crop_id, top_k=5):
    if crop_id in crop_id_to_index:
        query_vector = embeddings[crop_id_to_index[crop_id]]
    elif crop_id in all_crop_id_to_path:
        # Para crops test: se codifica la imagen como query, pero NO se indexa.
        query_vector = encode_image_crop(all_crop_id_to_path[crop_id])
    else:
        raise KeyError(f"Unknown crop_id: {crop_id}")
    return format_hits(qdrant_search(query_vector, top_k=top_k))

In [6]:
retrieve_by_class_name("gun", top_k=5)

,rank,score,crop_id,split,image_path,crop_path,class_id,class_name,x1,y1,x2,y2,bbox_area,relative_area
0,1,0.321751,train_042372_jpg.rf.ed9dcc93d3583accc3a174959b...,train,C:\Users\Juan Ramirez\.cache\kagglehub\dataset...,data\crops\train\train_042372_jpg.rf.ed9dcc93d...,4,wrench,148,163,183,231,2380,0.013753
1,2,0.316231,valid_017783_jpg.rf.7b537db13a7b4f643415e6208f...,valid,C:\Users\Juan Ramirez\.cache\kagglehub\dataset...,data\crops\valid\valid_017783_jpg.rf.7b537db13...,0,gun,236,197,268,244,1504,0.008691
2,3,0.312220,train_040517_jpg.rf.59b312f980fec4ec101165a33d...,train,C:\Users\Juan Ramirez\.cache\kagglehub\dataset...,data\crops\train\train_040517_jpg.rf.59b312f98...,1,knife,117,200,189,235,2520,0.014562
3,4,0.309794,train_041862_jpg.rf.bcf1d78e307e68de2b4ba10a68...,train,C:\Users\Juan Ramirez\.cache\kagglehub\dataset...,data\crops\train\train_041862_jpg.rf.bcf1d78e3...,1,knife,209,164,274,204,2600,0.015024
4,5,0.309350,valid_037531_jpg.rf.2b0f237fe8cc0560d4bfbcb690...,valid,C:\Users\Juan Ramirez\.cache\kagglehub\dataset...,data\crops\valid\valid_037531_jpg.rf.2b0f237fe...,1,knife,245,211,303,249,2204,0.012736


In [7]:
example_crop_id = all_metadata[all_metadata["split"].eq("test")].iloc[0]["crop_id"] if (all_metadata["split"] == "test").any() else metadata.iloc[0]["crop_id"]
retrieve_by_image(example_crop_id, top_k=5)

,rank,score,crop_id,split,image_path,crop_path,class_id,class_name,x1,y1,x2,y2,bbox_area,relative_area
0,1,0.956098,train_009704_jpg.rf.9ef53ed1c85ba054122870ae80...,train,C:\Users\Juan Ramirez\.cache\kagglehub\dataset...,data\crops\train\train_009704_jpg.rf.9ef53ed1c...,2,pliers,272,123,296,226,2472,0.014284
1,2,0.943103,train_026351_jpg.rf.4bca5c785e5ef43af7e7a8d07c...,train,C:\Users\Juan Ramirez\.cache\kagglehub\dataset...,data\crops\train\train_026351_jpg.rf.4bca5c785...,3,scissors,195,164,217,244,1760,0.010170
2,3,0.940430,train_009726_jpg.rf.624f5b7a8176759383a6230fcd...,train,C:\Users\Juan Ramirez\.cache\kagglehub\dataset...,data\crops\train\train_009726_jpg.rf.624f5b7a8...,2,pliers,233,135,256,249,2622,0.015151
3,4,0.924631,train_035220_jpg.rf.240bd5c5722161c172e292e969...,train,C:\Users\Juan Ramirez\.cache\kagglehub\dataset...,data\crops\train\train_035220_jpg.rf.240bd5c57...,3,scissors,182,140,201,225,1615,0.009332
4,5,0.920814,train_035519_jpg.rf.22779c2884434c9f9cc278493c...,train,C:\Users\Juan Ramirez\.cache\kagglehub\dataset...,data\crops\train\train_035519_jpg.rf.22779c288...,3,scissors,329,161,347,252,1638,0.009465
